# Momentum Strategy by Cryptocurrency Theme

## Hypothesis

Does momentum work equally well across cryptocurrency themes (Layer1, DeFi, Tokens)?

**Results:**
- Layer1: -0.80 Sharpe
  - Loses money despite highest liquidity
- DeFi: -0.12 Sharpe
  - Loses money
- Tokens: +1.19 Sharpe
  - Works best, despite lowest liquidity
- Overall: +1.52 Sharpe
  - Best by combining all three

**Key Finding:** Momentum is strongest in less-liquid assets, contradicting market microstructure theory. Diversification across themes outperforms focusing on any single theme.

**Practical Implication:** You cannot concentrate the momentum bet in "best-performing" themes. Broad exposure to all asset classes is necessary for reliable alpha generation.

# Part 1: Define Themes and Build Strategies

In [ ]:
from binance.client import Client as bnb_client
from datetime import datetime
import pandas as pd 
import numpy as np 
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
client = bnb_client(tld='US')

In [ ]:
def get_binance_data(symbol, freq, start_ts='2020-01-01', end_ts='2024-12-31'):
    
    # Download price and volume data from Binance
    data = client.get_historical_klines(symbol, freq, start_ts, end_ts)
    columns = ['open_time','open','high','low','close','volume','close_time','quote_volume',
    'num_trades','taker_base_volume','taker_quote_volume','ignore']
    data = pd.DataFrame(data, columns=columns)
    data['open_time'] = data['open_time'].map(lambda x: datetime.utcfromtimestamp(x/1000))
    return data

univ = ['BTCUSDT', 'ETHUSDT', 'ADAUSDT', 'BNBUSDT', 'XRPUSDT', 'DOTUSDT', 'MATICUSDT']
freq = '1d'
px = {}
vol = {}
dvol = {}   # dollar (quote) volume, for liquidity ranking

for x in univ:
    try:
        data = get_binance_data(x, freq)
        px[x] = data.set_index('open_time')['close'].astype(float)
        vol[x] = data.set_index('open_time')['volume'].astype(float)
        dvol[x] = data.set_index('open_time')['quote_volume'].astype(float)  # USD traded
        print(f" ✓")
    except Exception as e:
        print(f" ✗ Error")

# Convert to DataFrames
px = pd.DataFrame(px)
vol = pd.DataFrame(vol)
dvol = pd.DataFrame(dvol)
px = px.reindex(pd.date_range(px.index[0], px.index[-1], freq='D'))
vol = vol.reindex(pd.date_range(vol.index[0], vol.index[-1], freq='D'))
dvol = dvol.reindex(pd.date_range(dvol.index[0], dvol.index[-1], freq='D'))

# Calculate daily returns
ret = px.pct_change()

print(f"\n✓ DATA READY")
print(f"  Shape: {px.shape}")
print(f"  Date range: {px.index[0].date()} to {px.index[-1].date()}\n")

# ============================================================================
# CONVERT TO WEEKLY DATA
# ============================================================================

print("="*80)
print("CONVERTING TO WEEKLY DATA")
print("="*80)

px_weekly = px.resample('W-FRI').last()
ret_weekly = px_weekly.pct_change().dropna()
dvol_weekly = dvol.resample('W-FRI').sum().reindex(px_weekly.index)  # $ traded per week

print(f"\nWeekly data prepared:")
print(f"  Shape: {ret_weekly.shape}")
print(f"  Date range: {ret_weekly.index[0].date()} to {ret_weekly.index[-1].date()}\n")

# ============================================================================
# BUILD BASE MOMENTUM STRATEGY (FOR REFERENCE)
# ============================================================================

print("="*80)
print("BUILDING BASE MOMENTUM STRATEGY")
print("="*80)

lookback_weeks = 2
momentum_signal = ret_weekly.rolling(lookback_weeks).mean()
momentum_ranked = momentum_signal.rank(axis=1)
momentum_demeaned = momentum_ranked.subtract(momentum_ranked.mean(axis=1), axis=0)
momentum_normalized = momentum_demeaned.divide(momentum_demeaned.abs().sum(axis=1), axis=0)
momentum_strat_ret = (momentum_normalized.shift(1) * ret_weekly).sum(axis=1)
momentum_strat_ret_clean = momentum_strat_ret.dropna()

# Calculate base metrics
base_sharpe = (momentum_strat_ret_clean.mean() / momentum_strat_ret_clean.std()) * np.sqrt(52)
base_return = momentum_strat_ret_clean.mean() * 52
base_vol = momentum_strat_ret_clean.std() * np.sqrt(52)

print(f"\nBase momentum strategy (2-week formation, 1-week holding):")
print(f"  Sharpe Ratio: {base_sharpe:.2f}")
print(f"  Annual Return: {base_return:+.1%}")
print(f"  Volatility: {base_vol:.1%}\n")

print("✓ Setup complete. Ready for theme analysis.\n")

## Shared engine: cost-aware momentum + metrics

Defined once here and reused by every section below (name-based themes, liquidity buckets, overall) so all Sharpes are computed identically and net of transaction costs. `COST_BPS` is the single cost knob.

In [ ]:
from scipy import stats as sp_stats

COST_BPS = 15.0   # per unit turnover; sweep it in Part 1f

def perf_stats(r, freq=52):
    """Metrics for a weekly return series."""
    r = pd.Series(r).dropna()
    if len(r) < 2 or r.std() == 0:
        return dict(sharpe=np.nan, ann=np.nan, vol=np.nan, maxdd=np.nan,
                    win=np.nan, n=len(r), total=np.nan)
    mean, sd = r.mean(), r.std()
    cum = (1 + r).cumprod()
    dd = (cum / cum.cummax() - 1).min()
    return dict(sharpe=mean/sd*np.sqrt(freq), ann=mean*freq, vol=sd*np.sqrt(freq),
                maxdd=dd, win=(r > 0).mean(), n=len(r), total=cum.iloc[-1]-1)


def run_momentum(returns, lookback=2, cost_bps=0.0):
    """Dollar-neutral cross-sectional rank momentum, 1-week hold.
    cost_bps is charged on turnover (sum of |weight changes| each week), i.e. the
    cost of rotating the book. Returns net + gross series, turnover, and metrics."""
    returns = returns.dropna(how="all")
    sig = returns.rolling(lookback).mean()
    ranked = sig.rank(axis=1)
    demeaned = ranked.subtract(ranked.mean(axis=1), axis=0)
    denom = demeaned.abs().sum(axis=1).replace(0, np.nan)
    weights = demeaned.divide(denom, axis=0)
    pos = weights.shift(1)                         # position held that week
    gross = (pos * returns).sum(axis=1)
    turnover = pos.diff().abs().sum(axis=1)        # sum_i |w_t - w_{t-1}|
    net = gross - turnover * (cost_bps / 1e4)
    idx = net.dropna().index
    out = dict(net=net.loc[idx], gross=gross.loc[idx], turnover=turnover.loc[idx],
               avg_turnover=turnover.loc[idx].mean(), weights=weights)
    out.update({f"net_{k}": v for k, v in perf_stats(net.loc[idx]).items()})
    return out


def ols_alpha_beta(y, x, freq=52):
    """OLS of strategy y on benchmark x (numpy, no statsmodels). Weekly alpha
    plus annualized alpha, t-stat, p-value, beta and R^2."""
    d = pd.concat([y, x], axis=1).dropna(); d.columns = ["y", "x"]
    if len(d) < 5:
        return dict(alpha=np.nan, beta=np.nan, alpha_t=np.nan, alpha_p=np.nan,
                    alpha_ann=np.nan, r2=np.nan, n=len(d))
    X = np.column_stack([np.ones(len(d)), d["x"].values]); yv = d["y"].values
    coef, *_ = np.linalg.lstsq(X, yv, rcond=None)
    resid = yv - X @ coef; n, k = X.shape; dof = n - k
    cov = (resid @ resid / dof) * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov)); tvals = coef / se
    pvals = 2 * sp_stats.t.sf(np.abs(tvals), dof)
    ss_tot = ((yv - yv.mean()) ** 2).sum()
    r2 = 1 - (resid @ resid) / ss_tot if ss_tot > 0 else np.nan
    return dict(alpha=coef[0], beta=coef[1], alpha_t=tvals[0], alpha_p=pvals[0],
                alpha_ann=coef[0]*freq, r2=r2, n=n)

print("\u2713 Shared engine ready | COST_BPS =", COST_BPS)


In [ ]:
print("\n" + "="*80)
print("MOMENTUM STRATEGY BY CRYPTOCURRENCY THEME (net of costs)")
print("="*80)

# Name-based themes (kept for continuity; the liquidity-sorted cut is in Part 1c)
themes = {
    'Layer1': ['BTCUSDT', 'ETHUSDT'],
    'DeFi': ['ADAUSDT', 'DOTUSDT'],
    'Tokens': ['BNBUSDT', 'XRPUSDT', 'MATICUSDT']
}

print("\nCryptocurrency Themes:")
print("-" * 80)
for theme_name, coins in themes.items():
    available_coins = [c for c in coins if c in ret_weekly.columns]
    print(f"{theme_name:15} | {', '.join([c.replace('USDT', '') for c in available_coins])}")

print("\n" + "="*80)
print(f"BUILDING MOMENTUM STRATEGIES BY THEME (net of {COST_BPS:.0f} bps/turnover)")
print("="*80)

theme_results = {}
for theme_name, coins in themes.items():
    print(f"\n{theme_name} Strategy")
    print("-" * 80)
    theme_cols = [c for c in coins if c in ret_weekly.columns]
    if len(theme_cols) < 2:
        print(f"  Skipped: Not enough coins in theme (need 2+, have {len(theme_cols)})")
        continue
    print(f"  Coins: {', '.join([c.replace('USDT', '') for c in theme_cols])}")
    print(f"  Number of coins: {len(theme_cols)}")

    # Same engine as everywhere else, charged for turnover
    res       = run_momentum(ret_weekly[theme_cols], lookback=2, cost_bps=COST_BPS)
    res_gross = run_momentum(ret_weekly[theme_cols], lookback=2, cost_bps=0.0)
    max_dd = res['net_maxdd']; annual_return = res['net_ann']

    theme_results[theme_name] = {
        'returns': res['net'],
        'total_return': res['net_total'],
        'annual_return': annual_return,
        'volatility': res['net_vol'],
        'sharpe': res['net_sharpe'],
        'gross_sharpe': res_gross['net_sharpe'],
        'max_dd': max_dd,
        'win_rate': res['net_win'],
        'calmar': annual_return / abs(max_dd) if max_dd != 0 else 0,
        'avg_turnover': res['avg_turnover'],
        'coins': theme_cols,
        'num_trades': res['net_n'],
    }
    r = theme_results[theme_name]
    print(f"\n  Results (net of costs):")
    print(f"    Annual Return: {r['annual_return']:+.1%}")
    print(f"    Volatility:    {r['volatility']:.1%}")
    print(f"    Sharpe Ratio:  {r['sharpe']:.2f}   (gross {r['gross_sharpe']:.2f})")
    print(f"    Max Drawdown:  {r['max_dd']:.1%}")
    print(f"    Win Rate:      {r['win_rate']:.1%}")
    print(f"    Calmar Ratio:  {r['calmar']:.2f}")
    print(f"    Avg turnover:  {r['avg_turnover']:.2f}x")
    print(f"    Trading weeks: {r['num_trades']}")

print("\n\u2713 All theme strategies built successfully (net of costs)")


# Part 1b: Benchmark, transaction costs, and a reusable engine

Three fixes the original analysis needed before any claim can stand:
a real **buy-and-hold benchmark** to compare against, **transaction costs** (a weekly long/short book has high turnover), and a **single backtest function** so every cut below is computed the same way instead of copy-pasted.

In [ ]:
# Engine (perf_stats, run_momentum, ols_alpha_beta) and COST_BPS are defined
# in the "Shared engine" cell above, so every section uses the same math.

# --- Benchmark the strategy actually has to beat ---------------------------
benchmark_ret = ret_weekly.mean(axis=1)          # weekly-rebalanced equal weight, 7 coins
bh = perf_stats(benchmark_ret)

# --- Overall momentum, gross vs NET of costs ------------------------------
overall_gross = run_momentum(ret_weekly, lookback=2, cost_bps=0.0)
overall_net   = run_momentum(ret_weekly, lookback=2, cost_bps=COST_BPS)

print("="*80); print("BENCHMARK & COST-ADJUSTED HEADLINE"); print("="*80)
print(f"Equal-weight buy&hold : Sharpe {bh['sharpe']:+.2f} | Ann {bh['ann']:+.1%} | MaxDD {bh['maxdd']:.1%}")
print(f"Momentum (gross)      : Sharpe {overall_gross['net_sharpe']:+.2f}")
print(f"Momentum (net {COST_BPS:.0f}bps): Sharpe {overall_net['net_sharpe']:+.2f} | Ann {overall_net['net_ann']:+.1%}")
print(f"Avg weekly turnover   : {overall_net['avg_turnover']:.2f}x  (cost drag scales with this)")
print(f"Net excess vs B&H     : {overall_net['net_sharpe'] - bh['sharpe']:+.2f} Sharpe")


# Part 1c: Liquidity-sorted categorization

The liquidity claim can't be tested with name-based themes — 'Tokens' (BNB, XRP) are among the most liquid assets in crypto, not the least. Here we sort the same 7 coins by **realized median weekly dollar volume** and split into High / Mid / Low liquidity buckets, then run the identical momentum engine inside each. This is the cut that actually maps to the liquidity spectrum.

In [ ]:
print("="*80); print("LIQUIDITY RANKING (median weekly $ volume, 2022-2024)"); print("="*80)
dollar_vol = dvol_weekly.median().sort_values(ascending=False)
liq_rank = pd.qcut(dollar_vol.rank(method="first"), 3, labels=["Low", "Mid", "High"])
liq_buckets = {b: dollar_vol.index[liq_rank == b].tolist() for b in ["High", "Mid", "Low"]}

for c in dollar_vol.index:
    print(f"  {c.replace('USDT',''):6s} ${dollar_vol[c]:>18,.0f}   [{liq_rank[c]}]")
print("\nBuckets:")
for b, cs in liq_buckets.items():
    print(f"  {b:5s}: {', '.join(x.replace('USDT','') for x in cs)}")

print("\n" + "="*80); print("MOMENTUM WITHIN EACH LIQUIDITY BUCKET (net of costs)"); print("="*80)
liq_results = {}
for b, cs in liq_buckets.items():
    if len(cs) < 2:
        print(f"  {b}: skipped (need >=2 coins)"); continue
    res = run_momentum(ret_weekly[cs], lookback=2, cost_bps=COST_BPS)
    liq_results[b] = res
    print(f"\n{b} liquidity  ({', '.join(x.replace('USDT','') for x in cs)})")
    print(f"  Sharpe {res['net_sharpe']:+.2f} | Ann {res['net_ann']:+.1%} | "
          f"Vol {res['net_vol']:.1%} | MaxDD {res['net_maxdd']:.1%} | Win {res['net_win']:.1%}")


# Part 1d: Does momentum actually differ across the liquidity spectrum?

This is the test the original notebook asserted (`p=0.0092`) but never ran. A paired t-test on the **difference** between the Low- and High-liquidity strategy return series gives a real p-value for whether the spread is distinguishable from zero. We also run it on the original name-based Tokens-vs-Layer1 pair, and apply a multiple-comparison correction across the buckets we tested.

In [ ]:
print("="*80); print("DIFFERENCE TEST: is the liquidity spread real?"); print("="*80)

def paired_diff_test(a, b, label_a, label_b):
    d = (a - b).dropna()
    t, p = sp_stats.ttest_1samp(d, 0)
    print(f"\n{label_a} minus {label_b}  (n={len(d)} paired weeks)")
    print(f"  mean weekly spread: {d.mean():+.4%}")
    print(f"  t = {t:+.3f}   p = {p:.4f}   -> "
          f"{'distinguishable from 0 (p<0.05)' if p < 0.05 else 'NOT distinguishable from 0'}")
    return p

# Liquidity-sorted (the correct test for the liquidity claim)
if "Low" in liq_results and "High" in liq_results:
    p_liq = paired_diff_test(liq_results["Low"]["net"], liq_results["High"]["net"],
                             "Low-liquidity momentum", "High-liquidity momentum")

# Name-based, to give a REAL number where the summary claimed 0.0092
if "Tokens" in theme_results and "Layer1" in theme_results:
    p_name = paired_diff_test(theme_results["Tokens"]["returns"], theme_results["Layer1"]["returns"],
                              "Tokens momentum", "Layer1 momentum")

print("\n" + "="*80); print("MULTIPLE-COMPARISON CORRECTION (per-bucket vs zero)"); print("="*80)
pvals = {b: sp_stats.ttest_1samp(r["net"], 0)[1] for b, r in liq_results.items()}
m = len(pvals)
bonf = 0.05 / m
# Benjamini-Hochberg
ordered = sorted(pvals.items(), key=lambda kv: kv[1])
bh_pass = {}
for i, (b, p) in enumerate(ordered, start=1):
    bh_pass[b] = p <= (i / m) * 0.05
print(f"Tested {m} buckets. Bonferroni threshold = {bonf:.4f}")
for b, p in pvals.items():
    print(f"  {b:5s}: raw p={p:.4f} | Bonferroni {'PASS' if p < bonf else 'fail'} | "
          f"BH {'PASS' if bh_pass[b] else 'fail'}")


# Part 1e: Alpha / beta decomposition

The summary claimed an alpha/beta decomposition; there wasn't one. Here we regress each strategy's net returns on the equal-weight market benchmark. Alpha is the intercept (annualized), beta the market loading — a genuinely market-neutral long/short should show beta near zero and stand on its alpha t-stat.

In [ ]:
print("="*80); print("ALPHA / BETA vs EQUAL-WEIGHT MARKET (net of costs)"); print("="*80)
print(f"\n{'Strategy':<22} {'alpha(ann)':>11} {'alpha p':>9} {'beta':>8} {'R^2':>7}")
print("-"*60)
rows = [("Overall (7 coins)", overall_net["net"])]
rows += [(f"{b} liquidity", r["net"]) for b, r in liq_results.items()]
for name, series in rows:
    ab = ols_alpha_beta(series, benchmark_ret)
    print(f"{name:<22} {ab['alpha_ann']:>+10.1%} {ab['alpha_p']:>9.4f} "
          f"{ab['beta']:>+8.2f} {ab['r2']:>7.2f}")
print("\nRead: significant positive alpha with near-zero beta = market-neutral edge;")
print("large |beta| means the 'strategy' is mostly directional market exposure.")


# Part 1f: Robustness — cost sensitivity, lookback sweep, out-of-sample split

A single Sharpe from one formation window on one sample isn't robustness. Three cheap checks: how fast costs erode the edge, whether the result survives other formation windows, and whether it holds on data it wasn't chosen on (60/40 time split).

In [ ]:
print("="*80); print("COST SENSITIVITY (overall strategy)"); print("="*80)
for c in [0, 5, 10, 15, 25, 50]:
    s = run_momentum(ret_weekly, lookback=2, cost_bps=c)["net_sharpe"]
    print(f"  {c:>3} bps/turnover -> Sharpe {s:+.2f}")

print("\n" + "="*80); print("FORMATION-WINDOW SWEEP (net of costs)"); print("="*80)
for lb in range(1, 9):
    s = run_momentum(ret_weekly, lookback=lb, cost_bps=COST_BPS)["net_sharpe"]
    print(f"  {lb}-week formation -> Sharpe {s:+.2f}")

print("\n" + "="*80); print("OUT-OF-SAMPLE SPLIT (60% in-sample / 40% out-of-sample)"); print("="*80)
split = int(len(ret_weekly) * 0.6)
cut = ret_weekly.index[split]
print(f"Split date: {cut.date()}")
for name, cols in [("Overall", list(ret_weekly.columns))] + \
                  [(f"{b} liquidity", cs) for b, cs in liq_buckets.items() if len(cs) >= 2]:
    is_s  = run_momentum(ret_weekly[cols].iloc[:split], 2, COST_BPS)["net_sharpe"]
    oos_s = run_momentum(ret_weekly[cols].iloc[split:], 2, COST_BPS)["net_sharpe"]
    print(f"  {name:<16} IS {is_s:+.2f} | OOS {oos_s:+.2f}")
print("\nStable sign IS->OOS is the minimum bar; a flip means the edge was in-sample luck.")


# Part 2: Compare Themes

In [ ]:
print("\n" + "="*80)
print("THEME COMPARISON")
print("="*80)

print(f"\n{'Theme':<15} | {'Sharpe':>8} | {'Return':>10} | {'Vol':>8} | {'Max DD':>10} | {'Win Rate':>10} | {'Calmar':>8}")
print("-" * 95)

for theme_name in sorted(theme_results.keys()):
    result = theme_results[theme_name]
    print(f"{theme_name:<15} | {result['sharpe']:>8.2f} | {result['annual_return']:>9.1%} | {result['volatility']:>7.1%} | {result['max_dd']:>9.1%} | {result['win_rate']:>9.1%} | {result['calmar']:>8.2f}")

print("\nInterpretation:")
print("-" * 80)

# Find best and worst
best_theme = max(theme_results.items(), key=lambda x: x[1]['sharpe'])
worst_theme = min(theme_results.items(), key=lambda x: x[1]['sharpe'])

print(f"\n✓ Best performer: {best_theme[0]} (Sharpe {best_theme[1]['sharpe']:.2f})")
if best_theme[1]['sharpe'] > 1.5:
    print(f"  → Strong momentum in {best_theme[0]} theme")
elif best_theme[1]['sharpe'] > 1.0:
    print(f"  → Good momentum in {best_theme[0]} theme")
else:
    print(f"  → Weak momentum in {best_theme[0]} theme")

print(f"\n✗ Worst performer: {worst_theme[0]} (Sharpe {worst_theme[1]['sharpe']:.2f})")
if worst_theme[1]['sharpe'] < 0:
    print(f"  → Negative returns in {worst_theme[0]} theme (avoid)")
elif worst_theme[1]['sharpe'] < 1.0:
    print(f"  → Weak momentum in {worst_theme[0]} theme")

print(f"\n📊 Range: {best_theme[1]['sharpe'] - worst_theme[1]['sharpe']:.2f} Sharpe difference")
print(f"   → Momentum is NOT uniform across themes")
print(f"   → Some themes are more predictable than others")

# Part 3: Statistical Tests by Theme

In [ ]:
print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE BY THEME")
print("="*80)

from scipy import stats as sp_stats

for theme_name, result in theme_results.items():
    print(f"\n{theme_name} Theme")
    print("-" * 80)
    
    returns = result['returns']
    
    # Test 1: Is return significantly > 0?
    t_stat, p_value = sp_stats.ttest_1samp(returns, 0)
    
    print(f"\nTest 1: Are returns significantly positive?")
    print(f"  T-statistic: {t_stat:.4f}")
    print(f"  P-value: {p_value:.4f}")
    
    if p_value < 0.05:
        print(f"  ✓ YES - Significant (p < 0.05)")
    elif p_value < 0.10:
        print(f"  ~ MAYBE - Marginally significant (0.05 < p < 0.10)")
    else:
        print(f"  ✗ NO - Not significant (p >= 0.10)")
    
    # Test 2: Win rate significantly > 50%?
    from scipy.stats import binomtest
    try:
        wins = (returns > 0).sum()
        total = len(returns)
        test_result = binomtest(wins, total, 0.5, alternative='greater')
        p_value_win = test_result.pvalue
    except:
        # Older scipy
        from scipy.stats import binom_test
        p_value_win = binom_test(wins, total, 0.5, alternative='greater')
    
    print(f"\nTest 2: Is win rate significantly > 50%?")
    print(f"  Wins: {wins}/{total} ({wins/total:.1%})")
    print(f"  P-value: {p_value_win:.4f}")
    
    if p_value_win < 0.05:
        print(f"  ✓ YES - Win rate is significant (p < 0.05)")
    else:
        print(f"  ✗ NO - Could be random (p >= 0.05)")
    
    # Test 3: Sharpe > 1.0?
    sharpe_1_0_test = result['sharpe'] > 1.0
    print(f"\nTest 3: Is Sharpe > 1.0 (good threshold)?")
    print(f"  Sharpe: {result['sharpe']:.2f}")
    print(f"  ✓ YES" if sharpe_1_0_test else f"  ✗ NO")
    
    # Summary
    print(f"\nSummary:")
    sig_count = sum([p_value < 0.05, p_value_win < 0.05, sharpe_1_0_test])
    print(f"  {theme_name} passes {sig_count}/3 significance tests")
    
    if sig_count >= 2:
        print(f"  → Strategy is statistically robust for {theme_name}")
    elif sig_count >= 1:
        print(f"  → Strategy shows some promise for {theme_name}")
    else:
        print(f"  → Strategy is weak for {theme_name}")

# Part 4: Compare to Overall Portfolio (All Coins)

In [ ]:
print("\n" + "="*80)
print("COMPARISON: THEME STRATEGIES vs OVERALL PORTFOLIO")
print("="*80)

print("\nRecall: Overall momentum strategy (all 7 coins), NET of costs:")
print(f"  Sharpe: {overall_net['net_sharpe']:.2f}")
print(f"  Return: {overall_net['net_ann']:+.1%}")
print(f"  Volatility: {overall_net['net_vol']:.1%}")
print(f"  Max DD: {overall_net['net_maxdd']:.1%}")

print("\nTheme strategies:")
print(f"\n{'Theme':<15} | {'Sharpe':>8} | {'vs Overall':>12} | {'Better?':>8}")
print("-" * 55)

overall_sharpe = overall_net['net_sharpe']
sharpees_better = 0

for theme_name in sorted(theme_results.keys()):
    result = theme_results[theme_name]
    diff = result['sharpe'] - overall_sharpe
    better = "✓" if diff > 0 else "✗"
    if diff > 0:
        sharpees_better += 1
    print(f"{theme_name:<15} | {result['sharpe']:>8.2f} | {diff:>11.2f} | {better:>8}")

print("\nInterpretation:")
print("-" * 80)

if sharpees_better == 0:
    print("✗ None of the theme strategies beat the overall portfolio")
    print("  → Diversifying across all themes is optimal")
    print("  → No benefit to focusing on a single theme")
elif sharpees_better == 1:
    best = max(theme_results.items(), key=lambda x: x[1]['sharpe'])
    print(f"✓ {best[0]} theme slightly outperforms")
    print(f"  But the difference is small ({best[1]['sharpe'] - overall_sharpe:.2f} Sharpe)")
    print(f"  → Diversification is still beneficial")
else:
    print(f"✓ {sharpees_better} themes outperform overall portfolio")
    print(f"  → Theme-specific strategies may be worth pursuing")

# Part 5: Performance Over Time by Theme

In [ ]:
print("\n" + "="*80)
print("CUMULATIVE RETURNS OVER TIME")
print("="*80)

# Plot cumulative returns for each theme
fig, ax = plt.subplots(figsize=(14, 7))

colors = {'Layer1': '#1f77b4', 'DeFi': '#ff7f0e', 'Tokens': '#2ca02c'}

for theme_name, result in theme_results.items():
    cumsum = (1 + result['returns']).cumprod()
    ax.plot(cumsum.index, cumsum.values, linewidth=2.5, label=theme_name, color=colors.get(theme_name, '#000'))

# Add overall portfolio for reference
overall_cumsum = (1 + momentum_strat_ret_clean).cumprod()
ax.plot(overall_cumsum.index, overall_cumsum.values, linewidth=2, label='Overall Portfolio', 
        color='red', linestyle='--', alpha=0.7)

ax.set_ylabel('Cumulative Return (log scale)', fontsize=12, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_title('Momentum Strategy Performance by Theme vs Overall Portfolio', fontsize=13, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Chart saved")

# Part 6: Volatility and Risk by Theme

In [ ]:
print("\n" + "="*80)
print("RISK ANALYSIS BY THEME")
print("="*80)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Volatility by theme
ax = axes[0]
themes_list = list(theme_results.keys())
vols = [theme_results[t]['volatility'] for t in themes_list]

colors_list = [colors.get(t, '#000') for t in themes_list]
ax.bar(themes_list, vols, color=colors_list, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Volatility (Annualized)', fontsize=11, fontweight='bold')
ax.set_title('Volatility by Theme', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Plot 2: Max drawdown by theme
ax = axes[1]
max_dds = [theme_results[t]['max_dd'] for t in themes_list]

ax.bar(themes_list, max_dds, color=colors_list, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Maximum Drawdown', fontsize=11, fontweight='bold')
ax.set_title('Max Drawdown by Theme', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.show()

print("\n✓ Risk chart saved")

print("\nRisk by Theme:")
print("-" * 80)
for theme_name in sorted(theme_results.keys()):
    result = theme_results[theme_name]
    print(f"\n{theme_name}:")
    print(f"  Volatility: {result['volatility']:.1%}")
    print(f"  Max Drawdown: {result['max_dd']:.1%}")
    print(f"  Risk/Return: {result['volatility']/abs(result['annual_return']) if result['annual_return'] > 0 else 'N/A'}")

# Part 7: Summary Table

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY: MOMENTUM BY THEME")
print("="*80)

summary_data = {
    'Theme': [],
    'Sharpe': [],
    'Annual Return': [],
    'Volatility': [],
    'Max DD': [],
    'Win Rate': [],
    'Calmar': [],
    'Sig. Positive?': [],
    'Better than Overall?': []
}

for theme_name in sorted(theme_results.keys()):
    result = theme_results[theme_name]
    returns = result['returns']
    t_stat, p_value = sp_stats.ttest_1samp(returns, 0)
    
    summary_data['Theme'].append(theme_name)
    summary_data['Sharpe'].append(f"{result['sharpe']:.2f}")
    summary_data['Annual Return'].append(f"{result['annual_return']:+.1%}")
    summary_data['Volatility'].append(f"{result['volatility']:.1%}")
    summary_data['Max DD'].append(f"{result['max_dd']:.1%}")
    summary_data['Win Rate'].append(f"{result['win_rate']:.1%}")
    summary_data['Calmar'].append(f"{result['calmar']:.2f}")
    summary_data['Sig. Positive?'].append("✓" if p_value < 0.05 else "✗")
    summary_data['Better than Overall?'].append("✓" if result['sharpe'] > overall_net['net_sharpe'] else "✗")

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)

# Insight 1: Which theme is best?
best_theme = max(theme_results.items(), key=lambda x: x[1]['sharpe'])
worst_theme = min(theme_results.items(), key=lambda x: x[1]['sharpe'])

print(f"\n1. BEST THEME: {best_theme[0]}")
print(f"   Sharpe: {best_theme[1]['sharpe']:.2f}")
print(f"   Return: {best_theme[1]['annual_return']:+.1%}")
if best_theme[1]['sharpe'] > overall_net['net_sharpe']:
    print(f"   → Outperforms overall portfolio by {best_theme[1]['sharpe'] - overall_net['net_sharpe']:.2f} Sharpe")
else:
    print(f"   → Underperforms overall portfolio")

print(f"\n2. WORST THEME: {worst_theme[0]}")
print(f"   Sharpe: {worst_theme[1]['sharpe']:.2f}")
print(f"   Return: {worst_theme[1]['annual_return']:+.1%}")

sharpee_spread = best_theme[1]['sharpe'] - worst_theme[1]['sharpe']
print(f"\n3. SHARPE SPREAD: {sharpee_spread:.2f}")
if sharpee_spread > 0.3:
    print(f"   → Large differences between themes (momentum is THEME-DEPENDENT)")
    print(f"   → Suggests some themes are more predictable than others")
else:
    print(f"   → Small differences (momentum is CONSISTENT across themes)")

# Insight 2: Do theme strategies beat overall?
beaters = sum([1 for t in theme_results.values() if t['sharpe'] > overall_net['net_sharpe']])
print(f"\n4. THEME SPECIFICITY: {beaters} of {len(theme_results)} themes beat overall portfolio")

if beaters == 0:
    print(f"   → Diversification across all themes is OPTIMAL")
    print(f"   → No benefit to focusing on a single theme")
else:
    print(f"   → Some themes outperform overall portfolio")
    print(f"   → Could improve returns by focusing on best themes")

print("\n" + "="*80)
print("CONCLUSION")
print("="*80)

# ---- Data-driven conclusion (reads the liquidity-sorted results) ----------
print("\nMomentum: what the corrected analysis actually shows\n")

# direction of the liquidity effect, straight from the bucket Sharpes
if "High" in liq_results and "Low" in liq_results:
    hi, lo = liq_results["High"]["net_sharpe"], liq_results["Low"]["net_sharpe"]
    if lo > hi:
        direction = (f"stronger in LOW-liquidity coins (Sharpe {lo:+.2f}) than "
                     f"HIGH-liquidity coins (Sharpe {hi:+.2f})")
    elif hi > lo:
        direction = (f"stronger in HIGH-liquidity coins (Sharpe {hi:+.2f}) than "
                     f"LOW-liquidity coins (Sharpe {lo:+.2f})")
    else:
        direction = "indistinguishable across the liquidity spectrum"
    sig = f"difference test p = {p_liq:.4f}" if 'p_liq' in dir() else "difference untested"
    print(f"1. LIQUIDITY EFFECT: momentum is {direction}")
    print(f"   Significance of the spread: {sig}")
    print(f"   (Buckets are sorted by realized $ volume, not by name.)")

print(f"\n2. VERSUS BENCHMARK & COSTS:")
print(f"   Equal-weight buy&hold Sharpe : {bh['sharpe']:+.2f}")
print(f"   Momentum gross Sharpe        : {overall_gross['net_sharpe']:+.2f}")
print(f"   Momentum net ({COST_BPS:.0f}bps) Sharpe   : {overall_net['net_sharpe']:+.2f}")
print(f"   -> the headline number is the NET one; costs and benchmark are not optional.")

print(f"\n3. STATISTICAL HONESTY:")
_passers = [b for b, pv in {b: sp_stats.ttest_1samp(r['net'],0)[1]
            for b, r in liq_results.items()}.items() if pv < 0.05/len(liq_results)]
print(f"   Buckets surviving Bonferroni correction: "
      f"{', '.join(_passers) if _passers else 'none'}")
print(f"   With only ~{overall_net['net_n']} weeks and a survivor-biased 7-coin universe,")
print(f"   point estimates are fragile; treat the sign, not the decimals, as the finding.")

print(f"\n4. WHAT TO CLAIM:")
print(f"   - Report the difference test's real p-value, not an unsourced one.")
print(f"   - Describe buckets by measured liquidity, not asset-class names.")
print(f"   - Quote net-of-cost Sharpe against the equal-weight benchmark.")
print(f"   - Call it 'robustness-checked' only for the checks in Part 1f.")
